# Full clean headline run — wave mode: one seed at a time, all 5 models

Each **wave** spawns 5 parallel jobs (one per family) for a single seed into a **shared, fixed
run directory**. Three waves = the complete 5×3×21 dataset. Why waves:
- **cost control** — each wave is ~1/3 of the total (~\$8–18 batched); stop any time
- **early science** — wave 0 alone yields a complete 5-family figure at n=1
- all 5 jobs fit your 10-GPU cap, so a wave is fully parallel (~1–3.5h wall)

**To run a wave:** set `SEED` in the config cell (`'0'`, later `'1'`, then `'2'`), Run all,
close the tab. `RUN_DIR` is a fixed name — do **not** change it between waves, that's what makes
the three waves one dataset. Fire-and-forget: jobs run server-side on the deployed app.

In [ ]:
!pip install -q modal
# Paste your (rotated!) Modal token line:
!modal token set --token-id ak-XXXX --token-secret as-XXXX --profile=willgray
!modal profile activate willgray

In [ ]:
import os
RUN_DIR = 'run-clean-1'   # FIXED across all three waves — do not change between waves
SEED    = '0'             # <-- set to '0', then '1', then '2' (one wave each)
os.environ['RUN_DIR'] = RUN_DIR
os.environ['SEEDS']   = SEED
print(f'wave: seed {SEED} x 5 families  ->  necessity-results:/{RUN_DIR}')

In [ ]:
# Deploy current pipeline code, then spawn this wave's 5 jobs (fire-and-forget)
!rm -rf /content/repo && git clone -q https://github.com/wrgr/socratic-scenarios /content/repo
!modal deploy /content/repo/experiments/modal_headline.py
!python /content/repo/experiments/modal_submit.py

## Status — transcripts banked so far in this run dir (each wave adds up to 105; full = 315)

In [ ]:
!modal volume ls necessity-results /$RUN_DIR 2>/dev/null | grep -c jsonl || echo '0 (dir not created yet)'

## Pull — after any wave (a wave-0 pull already gives the 5-family n=1 figure)

In [ ]:
!mkdir -p /content/pull && modal volume get necessity-results /$RUN_DIR /content/pull --force
!find /content/pull -name 'dose_factqa_*_a*.jsonl' | wc -l
!cd /content && zip -qr factqa_$RUN_DIR.zip pull
from google.colab import drive; drive.mount('/content/drive')
import os, shutil
os.makedirs('/content/drive/MyDrive/necessity-audit', exist_ok=True)
shutil.copy(f"/content/factqa_{os.environ['RUN_DIR']}.zip", '/content/drive/MyDrive/necessity-audit/')
print('zip in Drive/necessity-audit — upload it to the session for scoring + paper update')